In [0]:
dbutils.widgets.dropdown("reset_tables", "no", ["no", "yes"])
reset_tables = dbutils.widgets.get("reset_tables")

In [0]:
# ---- settings and small helpers (the same block is in every notebook) ----
import json, os, re, uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

CATALOG = "workspace"      # change these two lines if your catalog / schema have other names
SCHEMA = "default"

def T(name):
    """Full table name, e.g. T("silver_quintile_index") -> workspace.default.silver_quintile_index"""
    return f"{CATALOG}.{SCHEMA}.{name}"

AUDIT_TABLE = T("pipeline_execution_logs")
QUARANTINE_TABLE = T("quarantine_records")

def now():
    """Current time (UTC)."""
    return datetime.now(timezone.utc).replace(tzinfo=None)

def sql_text(text):
    """Makes a value safe inside a SQL string: O'Brien -> O''Brien"""
    return str(text).replace("'", "''")

def norm_date(text):
    """Turns 1.10.2026, 01-10-2026 or 2026-10-01 into 01.10.2026"""
    for pattern in ["%d.%m.%Y", "%d-%m-%Y", "%Y-%m-%d"]:
        try:
            return datetime.strptime(str(text).strip(), pattern).strftime("%d.%m.%Y")
        except ValueError:
            pass
    raise ValueError(f"Cannot read the date '{text}'. Use dd.mm.yyyy, for example 01.10.2026")

# ---- execution log: one row per layer, per file/week, per run ----
AUDIT_COLUMNS = ("log_id string, batch_id string, load_type string, layer string, file_processed string, "
                 "target_table string, start_time timestamp, end_time timestamp, status string, "
                 "rows_read long, rows_inserted long, rows_updated long, rows_quarantined long, "
                 "error_message string, message string")

def log_audit(batch_id, load_type, layer, file_processed, target_table, start_time, status,
              rows_read=0, rows_inserted=0, rows_updated=0, rows_quarantined=0, message=""):
    """status is Success, Warning (success with notes), Skipped or Failure."""
    message = (message or "")[:1500]
    error_message = message if status == "Failure" else None      # error_message is only filled for failures
    row = (str(uuid.uuid4()), batch_id, load_type, layer, str(file_processed), target_table,
           start_time, now(), status, int(rows_read), int(rows_inserted), int(rows_updated),
           int(rows_quarantined), error_message, message)
    spark.createDataFrame([row], AUDIT_COLUMNS).write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

# ---- quarantine: rows (or whole files) that failed a check, saved together with the reason ----
def quarantine_from_list(items):
    """items = list of (record_ref, reason, record_json)"""
    return spark.createDataFrame(items, "record_ref string, reason string, record_json string")

def save_quarantine(df, layer, source_key, batch_id):
    """Replaces the earlier quarantine rows of the same layer and file, so re-runs never pile up duplicates.
    df needs the columns record_ref, reason and record_json."""
    spark.sql(f"DELETE FROM {QUARANTINE_TABLE} WHERE layer = '{layer}' AND source_file = '{sql_text(source_key)}'")
    rows = df.select(F.expr("uuid()").alias("quarantine_id"), F.lit(batch_id).alias("batch_id"),
                     F.lit(layer).alias("layer"), F.lit(source_key).alias("source_file"),
                     "record_ref", "reason", "record_json", F.current_timestamp().alias("quarantined_at"))
    rows.write.format("delta").mode("append").saveAsTable(QUARANTINE_TABLE)


In [0]:
# The columns of every table. Bronze keeps text (STRING); Silver has real types.
grid_columns = ", ".join(f"c{i:02d} STRING" for i in range(25))      # c00 STRING, c01 STRING, ... c24 STRING

pbs_bronze_columns = ("week_ending_date STRING, source_file STRING, sheet_name STRING, row_idx INT, "
                      + grid_columns +
                      ", extra_json STRING, batch_id STRING, load_type STRING, load_timestamp TIMESTAMP")

tables = {
    # ---- operations ----
    "pipeline_execution_logs": AUDIT_COLUMNS,
    "quarantine_records": ("quarantine_id STRING, batch_id STRING, layer STRING, source_file STRING, "
                           "record_ref STRING, reason STRING, record_json STRING, quarantined_at TIMESTAMP"),

    # ---- bronze: raw data as text + where it came from ----
    "bronze_wfp_raw": ("adm0_id STRING, adm0_name STRING, adm1_id STRING, adm1_name STRING, mkt_id STRING, "
                       "mkt_name STRING, cm_id STRING, cm_name STRING, cur_id STRING, cur_name STRING, "
                       "pt_id STRING, pt_name STRING, um_id STRING, um_name STRING, mp_month STRING, "
                       "mp_year STRING, mp_price STRING, mp_commoditysource STRING, extra_json STRING, "
                       "source_file STRING, source_file_signature STRING, batch_id STRING, load_type STRING, "
                       "load_timestamp TIMESTAMP"),
    "bronze_pbs_report": pbs_bronze_columns,
    "bronze_pbs_annex": pbs_bronze_columns,

    # ---- silver: clean, typed data ----
    "silver_wfp_price_observation": ("country STRING, admin_region STRING, market STRING, commodity_name STRING, "
                                     "category STRING, unit STRING, price_type STRING, price DOUBLE, "
                                     "currency STRING, price_date DATE, is_revised_flag BOOLEAN, "
                                     "ingestion_batch_id STRING, source_file STRING, load_timestamp TIMESTAMP, "
                                     "last_updated_timestamp TIMESTAMP"),
    "silver_quintile_index": ("quintile STRING, income_band STRING, index_value DOUBLE, index_prev_week DOUBLE, "
                              "index_prev_year DOUBLE, pct_change_wow DOUBLE, pct_change_yoy DOUBLE, "
                              "week_ending_date DATE, source_file STRING, batch_id STRING, "
                              "load_timestamp TIMESTAMP"),
    "silver_item_weight_impact": ("week_ending_date DATE, item_name STRING, unit STRING, price_direction STRING, "
                                  "price_current DOUBLE, price_prev_week DOUBLE, price_prev_year DOUBLE, "
                                  "pct_change_wow DOUBLE, pct_change_yoy DOUBLE, weight_lowest DOUBLE, "
                                  "weight_combined DOUBLE, impact_lowest DOUBLE, impact_combined DOUBLE, "
                                  "source_file STRING, batch_id STRING, load_timestamp TIMESTAMP"),
    "silver_city_price_observation": ("week_ending_date DATE, item_sl INT, item_name STRING, unit STRING, "
                                      "city STRING, city_code STRING, min_price DOUBLE, avg_price DOUBLE, "
                                      "max_price DOUBLE, price_dispersion_pct DOUBLE, source_file STRING, "
                                      "batch_id STRING, load_timestamp TIMESTAMP"),
}

# Optional reset: drop everything first
if reset_tables == "yes":
    for name in tables:
        spark.sql(f"DROP TABLE IF EXISTS {T(name)}")
        print("dropped", T(name))

# Create the tables that do not exist yet
for name, columns in tables.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {T(name)} ({columns}) USING DELTA")
    print("ready:", T(name))

# A log table from an older version may not have the 'message' column yet: add it (no data is lost)
log_columns = [c.lower() for c in spark.table(AUDIT_TABLE).columns]
if "message" not in log_columns:
    spark.sql(f"ALTER TABLE {AUDIT_TABLE} ADD COLUMNS (message STRING)")

display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}"))
dbutils.notebook.exit(json.dumps({"status": "Success", "reset": reset_tables}))